# Reflexion: Self-Reflecting Agents with LangGraph
## Generate, Critique, Revise — The Reflexion Loop
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/18-self-reflecting-agent/reflexion_workbook.ipynb)

**Reflexion** (Shinn et al., 2023) is an agent architecture in which a language model improves its own output by evaluating and critiquing it — iterating until it reaches high confidence or a safety cap. Unlike RAG, Reflexion requires no external knowledge store: the model's own reasoning is both the generator and the judge.

By the end of this workshop you will understand *why* Reflexion works, *how* each component fits together, and *how* to build and extend the loop in LangGraph from scratch.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — What is Reflexion and why does it exist? |
| 2 | **Structured Output** — Forcing the model to return typed critiques |
| 3 | **State Design** — Defining `ReflexionState` as a `TypedDict` |
| 4 | **Graph Nodes** — `generate`, `critique`, and the router |
| 5 | **Compile and Visualise** — Rendering the loop with Mermaid |
| 6 | **Run the Loop** — Easy questions, hard questions, streaming |
| 7 | **Debugging and Tuning** — Iteration counts, diminishing returns |
| ★ | **Extensions** — Score history, checkpointing, grounded Reflexion |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Shinn, N., Cassano, F., Labash, A., Gopalan, A., Narasimhan, K., & Yao, S. (2023). *Reflexion: Language Agents with Verbal Reinforcement Learning.* NeurIPS 2023. https://arxiv.org/abs/2303.11366
>
> Yao, S., Zhao, J., Yu, D., Du, N., Shafran, I., Narasimhan, K., & Cao, Y. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
>
> LangGraph documentation: https://langchain-ai.github.io/langgraph/

In [ ]:
# Install dependencies -- runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain-core",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected -- skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon) as OPENAI_API_KEY
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED -- add your key before running LLM cells.")
    print("  Local : echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab : Secrets panel -> Add secret 'OPENAI_API_KEY'")

---

## Part 1 — What Is Reflexion and Why Does It Exist?

---

### The Problem with Single-Pass Generation

Standard LLM pipelines generate an answer once and return it. This works well for factual lookups but fails on tasks that benefit from deliberation:
- Multi-step reasoning where early mistakes compound
- Questions where the first phrasing is incomplete or imprecise
- Code generation where the first attempt may have bugs
- Tasks where "good enough on the first try" is not good enough

**Reflexion** addresses this by adding a self-evaluation loop between generation and response. The agent drafts an answer, critiques it, and revises — looping until it is confident or a safety cap is reached.

---

### Three Approaches Compared

| Approach | External memory | Self-evaluation | Iterates | Best for |
|----------|----------------|-----------------|----------|----------|
| **Single-pass** | No | No | No | Simple factual Q&A |
| **ReAct** (Yao et al., 2023) | Tool calls | Implicit via observation | Per tool | Multi-step tool use |
| **Reflexion** (Shinn et al., 2023) | Verbal memory | Explicit self-critique | Per answer | Reasoning, writing, code |
| **Self-RAG** | Retrieval corpus | Critic tokens | Per token | Factual grounding |

Reflexion adds **verbal reinforcement**: the model stores its own critiques as natural language "memory" and uses them in the next revision pass — no gradient updates, no retraining.

---

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Actor** | The LLM component that generates or revises an answer |
| **Evaluator** | The LLM component (or separate model) that scores the answer |
| **Self-Reflection** | The critique text the evaluator produces, stored in state |
| **Verbal memory** | The accumulation of critique strings passed into future prompts |
| **Confidence score** | A structured label (`low` / `medium` / `high`) that drives the routing decision |
| **Stopping condition** | `confident == True` OR `iterations >= MAX_ITERATIONS` |
| **Structured output** | A Pydantic model the LLM is forced to populate — no string parsing |
| **TypedDict state** | The LangGraph state bag shared across all nodes in the graph |

### The Reflexion Loop

```
┌─────────────────────────────────────────────────────────────────────┐
│                        REFLEXION LOOP                               │
│                                                                     │
│                          START                                      │
│                            │                                        │
│                            ▼                                        │
│              ┌─────────────────────────┐                            │
│              │         ACTOR           │  <- answer_prompt           │
│              │  (generate / revise)    │     or revise_prompt        │
│              └────────────┬────────────┘     depending on iteration  │
│                           │                                         │
│                    answer written                                   │
│                    to state                                         │
│                           │                                         │
│                           ▼                                         │
│              ┌─────────────────────────┐                            │
│              │       EVALUATOR         │  <- critique_prompt         │
│              │   (structured output)   │     returns Confidence:     │
│              └────────────┬────────────┘     { score, critique }     │
│                           │                                         │
│               critique + score written                              │
│               to state                                              │
│                           │                                         │
│                  ┌────────┴────────┐                                │
│                  │    ROUTER       │                                 │
│                  │ should_continue │                                 │
│                  └───┬────────┬───┘                                 │
│                      │        │                                     │
│           score=high │        │ score=low/medium                    │
│    OR iterations=MAX │        │ AND iterations<MAX                  │
│                      │        │                                     │
│                      ▼        └──────────────────┐                  │
│                    END                           │                  │
│                                                  ▼                  │
│                                         ACTOR (revise)              │
│                                         <- critique in context       │
│                                                  │                  │
│                                    (loop back to EVALUATOR)         │
└─────────────────────────────────────────────────────────────────────┘
```

> **Source**: Architecture adapted from Shinn et al. (2023) — the original Reflexion paper, extended to show the LangGraph conditional edge routing.

### Reflection Strategies and Stopping Conditions

| Strategy | What the evaluator checks | Typical use |
|----------|--------------------------|-------------|
| **Confidence scoring** | Is the answer correct and complete? | General Q&A (this notebook) |
| **Factual critique** | Are specific claims verifiable? | Research tasks |
| **Test-based critique** | Does the code pass tests? | Code generation (Reflexion paper) |
| **Rubric scoring** | Does the answer meet explicit criteria? | Writing, essays |

| Stopping condition | Pros | Cons |
|-------------------|------|------|
| **Score threshold** (`high`) | Exits early when quality is reached | LLM may rate itself too high |
| **Max iterations** | Guarantees termination, cost-bounded | May exit before optimal quality |
| **Combined** (both) | Balances quality and cost | Requires tuning both parameters |

---

## Part 2 — Structured Output: Forcing Typed Critiques

---

### Why Not Parse the Critique from a String?

The routing decision (`should_continue`) needs a machine-readable confidence score. Parsing a string like `"I rate this as medium confidence"` is fragile — the model may vary its wording. Instead we use `with_structured_output`, which instructs the model to populate a Pydantic model.

```python
class Confidence(BaseModel):
    score: Literal["low", "medium", "high"]   # constrained to 3 values
    critique: str                               # free-text explanation

evaluator = llm.with_structured_output(Confidence)
result = evaluator.invoke(...)
result.score     # -> "low" | "medium" | "high"  (never fails parsing)
result.critique  # -> "The answer is missing X and Y..."
```

Under the hood, `with_structured_output` uses OpenAI function calling / JSON mode to guarantee the schema is respected.

In [ ]:
# ─── 2-A: Imports and LLM setup ───────────────────────────────────────────────

from typing import Annotated, Literal, TypedDict
import operator

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel


class Confidence(BaseModel):
    """Structured critique and confidence score for an answer."""

    score: Literal["low", "medium", "high"]
    critique: str


llm = ChatOpenAI(model="gpt-4o-mini")
evaluator = llm.with_structured_output(Confidence)

print("LLM and evaluator ready.")
print(f"  LLM model   : {llm.model_name}")
print(f"  Evaluator   : {type(evaluator).__name__} -> Confidence")

In [ ]:
# ─── 2-B: Define the three prompts that drive the loop ───────────────────────
#
# answer_prompt  -- used on the FIRST pass (no prior answer or critique)
# revise_prompt  -- used on ALL SUBSEQUENT passes (answer + critique in context)
# critique_prompt -- evaluates the current answer; returns Confidence

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question thoroughly and accurately."),
        ("human", "{question}"),
    ]
)

revise_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "Improve your answer based on the critique below.\n\n"
                "Original answer:\n{answer}\n\nCritique:\n{critique}\n\n"
                "Write a better answer that addresses every point raised."
            ),
        ),
        ("human", "{question}"),
    ]
)

critique_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "Evaluate this answer. Identify factual errors, missing information, "
                "or unclear explanations. "
                "Then rate your confidence that the answer is now correct and complete: "
                "low / medium / high."
            ),
        ),
        ("human", "Question: {question}\n\nAnswer: {answer}"),
    ]
)

print("Three prompts defined:")
print("  answer_prompt   -- first-pass generation")
print("  revise_prompt   -- revision with critique context")
print("  critique_prompt -- structured evaluation -> Confidence")

In [ ]:
# ─── 2-C: Smoke test the confidence grader ────────────────────────────────────
# Before building the graph, verify with_structured_output returns a proper
# Confidence object on both a good and a bad answer.

test_q = "What is LangGraph?"

test_cases = [
    (
        "Correct answer",
        "LangGraph is a Python library by LangChain for building stateful, "
        "multi-actor LLM applications using a graph of nodes and edges.",
    ),
    (
        "Incorrect answer",
        "LangGraph is a graph plotting library for data science and machine "
        "learning visualisations.",
    ),
]

for label, ans in test_cases:
    result = (critique_prompt | evaluator).invoke({"question": test_q, "answer": ans})
    print(f"{label}:")
    print(f"  score    = {result.score!r}")
    print(f"  critique = {result.critique[:160]}")
    print()

print("Observation: the correct answer should score 'medium' or 'high';")
print("the incorrect answer should score 'low' with a specific critique.")

---

## Part 3 — State Design: `ReflexionState`

---

### What Is LangGraph State?

In LangGraph, every node in the graph reads from and writes to a shared **state dictionary**. The state is defined as a `TypedDict` — a Python type annotation that specifies which keys exist and what type each holds.

Every node returns a dict with *only the keys it wants to update*. LangGraph merges those updates into the current state automatically.

```
State before generate:
  { question: "...", answer: "", critique: "", iterations: 0, confident: False }
                               ^
                        generate() returns { answer: "...", iterations: 1 }

State after generate:
  { question: "...", answer: "...", critique: "", iterations: 1, confident: False }
```

### State Field Reference

| Field | Type | Set by | Purpose |
|-------|------|--------|---------|
| `question` | `str` | Caller (initial input) | Original question — never mutated |
| `answer` | `str` | `generate` node | Current best answer; replaced each iteration |
| `critique` | `str` | `critique` node | Critique text from the last evaluation pass |
| `iterations` | `int` | `generate` node | Number of `generate` calls completed |
| `confident` | `bool` | `critique` node | Set `True` when the evaluator scores `high` |

In [ ]:
# ─── 3-A: Define the state and the max-iterations cap ─────────────────────────

MAX_ITERATIONS = 3


class ReflexionState(TypedDict):
    question: str
    answer: str
    critique: str
    iterations: int
    confident: bool


def make_initial_state(question: str) -> ReflexionState:
    """Helper: build the initial state dict for a new question."""
    return {
        "question": question,
        "answer": "",
        "critique": "",
        "iterations": 0,
        "confident": False,
    }


print(f"ReflexionState defined. MAX_ITERATIONS = {MAX_ITERATIONS}")
print()
print("Initial state for a sample question:")
sample_state = make_initial_state("What is the Reflexion technique?")
for k, v in sample_state.items():
    print(f"  {k:12} = {v!r}")

---

## Part 4 — Graph Nodes: `generate`, `critique`, and the Router

---

### Node Anatomy

Each LangGraph node is a plain Python function with the signature:

```python
def node_name(state: ReflexionState) -> dict:
    # read from state
    # do work
    return {"key_to_update": new_value}  # partial update
```

The router is different — it is a **conditional edge function** that returns the name of the next node (or `END`):

```python
def should_continue(state: ReflexionState) -> Literal["generate", "__end__"]:
    if state["confident"] or state["iterations"] >= MAX_ITERATIONS:
        return END
    return "generate"
```

### Dual-Mode Generate

The `generate` node handles both the **first draft** and all **revisions** in a single function:

```
if state["answer"] is empty  ->  use answer_prompt  (first pass)
if state["answer"] non-empty ->  use revise_prompt  (revision pass)
```

This avoids duplicating graph nodes and keeps the logic co-located.

In [ ]:
# ─── 4-A: Define generate, critique, and the routing function ─────────────────


def generate(state: ReflexionState) -> dict:
    """First-pass generation OR revision depending on whether an answer exists."""
    if state["answer"]:
        # Revision pass: pass previous answer + critique into the revise prompt
        response = (revise_prompt | llm).invoke(
            {
                "question": state["question"],
                "answer": state["answer"],
                "critique": state["critique"],
            }
        )
    else:
        # First pass: generate fresh answer with no prior context
        response = (answer_prompt | llm).invoke({"question": state["question"]})

    return {"answer": response.content, "iterations": state["iterations"] + 1}


def critique(state: ReflexionState) -> dict:
    """Evaluate the current answer; return critique text and confidence flag."""
    result = (critique_prompt | evaluator).invoke(
        {
            "question": state["question"],
            "answer": state["answer"],
        }
    )
    return {"critique": result.critique, "confident": result.score == "high"}


def should_continue(state: ReflexionState) -> Literal["generate", "__end__"]:
    """Route back to generate if not yet confident and cap not reached."""
    if state["confident"] or state["iterations"] >= MAX_ITERATIONS:
        return END
    return "generate"


print("Three graph functions defined:")
print("  generate(state)        -> dict with 'answer' and 'iterations'")
print("  critique(state)        -> dict with 'critique' and 'confident'")
print("  should_continue(state) -> 'generate' | END")

In [ ]:
# ─── 4-B: Unit-test each node in isolation before wiring the graph ────────────
# This is the most important debugging step: confirm each function works
# before composing them into a graph.

test_state: ReflexionState = make_initial_state("What is the Reflexion technique?")

print("--- Testing generate (first pass) ---")
gen_result = generate(test_state)
print(f"  iterations : {gen_result['iterations']}")
print(f"  answer     : {gen_result['answer'][:200]}...")
print()

# Merge the generate result into our test state
test_state = {**test_state, **gen_result}

print("--- Testing critique ---")
crit_result = critique(test_state)
print(f"  confident  : {crit_result['confident']}")
print(f"  critique   : {crit_result['critique'][:200]}...")
print()

# Merge critique result
test_state = {**test_state, **crit_result}

print("--- Testing should_continue ---")
route = should_continue(test_state)
print(f"  route: {route!r}  (expected 'generate' or END)")

---

## Part 5 — Compile the Graph and Visualise the Loop

---

### How the Graph Is Wired

LangGraph uses a `StateGraph` builder. You add nodes, then declare edges:

```python
graph.add_edge(START, "generate")           # entry point
graph.add_edge("generate", "critique")      # always runs after generate
graph.add_conditional_edges(                # router decides what comes after critique
    "critique",
    should_continue,
)  # maps 'generate' -> 'generate' and END -> END
```

The **conditional edge** is what creates the loop. LangGraph evaluates `should_continue(state)` after every `critique` call and routes accordingly. The graph compiles to an `app` object with `.invoke()`, `.stream()`, and `.get_graph()` methods.

In [ ]:
# ─── 5-A: Compile the Reflexion graph ─────────────────────────────────────────

graph = StateGraph(ReflexionState)

graph.add_node("generate", generate)
graph.add_node("critique", critique)

graph.add_edge(START, "generate")
graph.add_edge("generate", "critique")
graph.add_conditional_edges("critique", should_continue)

app = graph.compile()

print("Reflexion graph compiled.")
print(f"  Nodes : {list(app.get_graph().nodes.keys())}")
print(f"  Edges : {len(list(app.get_graph().edges))} total")

In [ ]:
# ─── 5-B: Visualise the graph ─────────────────────────────────────────────────
# The loop back-edge from critique -> generate should be clearly visible.

try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph image unavailable ({e}). Falling back to Mermaid text:")
    print()
    print(app.get_graph().draw_mermaid())

---

## Part 6 — Running the Loop

---

### What to Observe

When you run the loop you should see two distinct patterns:

**Easy questions** — questions with well-known factual answers
- The model usually scores `medium` or `high` on the first pass
- The loop typically exits in 1-2 iterations with `confident = True`

**Hard / unanswerable questions** — questions with no correct answer or outside the model's knowledge
- The evaluator continues scoring `low` or `medium` because no revision can produce a truly correct answer
- The loop runs to `MAX_ITERATIONS` and exits with `confident = False`
- This is the **safety cap** in action — the agent knows when to stop trying

The `iterations` and `confident` fields in the final state tell you which stopping condition fired.

In [ ]:
# ─── 6-A: Sample question bank ────────────────────────────────────────────────

QUESTIONS = {
    "easy": [
        "What are the main advantages of LangGraph over vanilla LangChain for building agents?",
        "Explain the difference between retrieval and generation in a RAG pipeline.",
        "What is the Reflexion technique and how does it improve LLM reasoning?",
        "How does human-in-the-loop checkpointing work in LangGraph?",
    ],
    "hard": [
        "What is the capital of the moon?",
        "How does LangGraph compare to CrewAI when running more than 100 agents concurrently in 2027?",
    ],
}

print("Question bank loaded:")
print(f"  {len(QUESTIONS['easy'])} easy questions (expect confident=True in 1-2 iterations)")
print(f"  {len(QUESTIONS['hard'])} hard questions (expect confident=False, hits MAX_ITERATIONS)")

In [ ]:
# ─── 6-B: Run easy questions ──────────────────────────────────────────────────
# These have clear, well-known answers.
# Expect confident = True in 1-2 iterations.

for q in QUESTIONS["easy"]:
    result = app.invoke(make_initial_state(q))
    short_answer = result["answer"][:280] + "..." if len(result["answer"]) > 280 else result["answer"]
    print(f"Q: {q}")
    print(f"   iters={result['iterations']}  confident={result['confident']}")
    print(f"   A: {short_answer}")
    print()

In [ ]:
# ─── 6-C: Run hard / unanswerable questions ───────────────────────────────────
# The agent will loop up to MAX_ITERATIONS and stop with confident = False,
# demonstrating the safety cap.

for q in QUESTIONS["hard"]:
    result = app.invoke(make_initial_state(q))
    short_answer = result["answer"][:280] + "..." if len(result["answer"]) > 280 else result["answer"]
    hit_cap = not result["confident"]
    print(f"Q: {q}")
    print(f"   iters={result['iterations']}  confident={result['confident']}  hit_cap={hit_cap}")
    print(f"   Final critique: {result['critique'][:160]}...")
    print(f"   A: {short_answer}")
    print()

In [ ]:
# ─── 6-D: Stream the loop step-by-step ───────────────────────────────────────
# stream_mode='updates' emits one dict per node as it completes.
# Watch the confidence score evolve (or plateau) across iterations.

stream_q = "What are the main advantages of LangGraph over vanilla LangChain for building agents?"
print(f'Streaming Reflexion for: "{stream_q}"\n')

for step in app.stream(make_initial_state(stream_q), stream_mode="updates"):
    node = list(step.keys())[0]
    out = step[node]
    print(f"[{node}]")
    if "iterations" in out:
        print(f"  iteration #{out['iterations']}")
    if "answer" in out:
        preview = out["answer"][:200]
        print(f"  answer: {preview}{'...' if len(out['answer']) > 200 else ''}")
    if "critique" in out:
        print(f"  critique: {out['critique'][:160]}")
    if "confident" in out:
        print(f"  confident: {out['confident']}")
    print()

---

## Part 7 — Debugging and Tuning the Loop

---

### Common Failure Modes

| Failure | Symptom | Root cause | Fix |
|---------|---------|-----------|-----|
| **Never exits** | Loop always hits `MAX_ITERATIONS` | Critique threshold too strict or evaluator overconfident | Check `critique_prompt` wording |
| **Exits too early** | Model rates itself `high` on iteration 1 | Critique too lenient, or question is genuinely easy | Lower threshold, add factual checks |
| **Revision makes it worse** | Answer quality degrades across iterations | Revise prompt loses context or introduces noise | Inspect revise prompt; consider keeping prior answer as reference |
| **Cost runaway** | Token usage spikes | `MAX_ITERATIONS` too high | Set a sensible cap (3-5 is typical) |
| **Structured output fails** | `ValidationError` on `Confidence` | Model did not follow schema | Switch to a model with reliable function calling (gpt-4o or above) |

---

### Debugging Checklist

1. **Stream before invoking** — watch the iteration-by-iteration state with `stream_mode='updates'`
2. **Unit-test each node** — call `generate(state)` and `critique(state)` directly before composing the graph
3. **Print the full prompts** — render the prompt template to see what the LLM receives
4. **Vary `MAX_ITERATIONS`** — try 1, 2, 5 and observe quality vs cost tradeoffs
5. **Log score history** — track `["low", "medium", "high"]` across iterations to see convergence

In [ ]:
# ─── 7-A: Inspect the rendered prompts the LLM actually receives ──────────────
# The most important debugging technique: print what the model sees.

debug_q = "What is the Reflexion technique and how does it improve LLM reasoning?"
debug_answer = "Reflexion is a type of neural network architecture."
debug_critique = (
    "The answer is incorrect. Reflexion is not a neural network architecture -- "
    "it is an agent prompting technique where the model critiques its own outputs."
)

print("=== ANSWER PROMPT (first pass) ===")
rendered_answer = answer_prompt.format_messages(question=debug_q)
for msg in rendered_answer:
    print(f"  [{msg.type.upper()}] {msg.content[:200]}")

print()
print("=== REVISE PROMPT (revision pass) ===")
rendered_revise = revise_prompt.format_messages(
    question=debug_q,
    answer=debug_answer,
    critique=debug_critique,
)
for msg in rendered_revise:
    print(f"  [{msg.type.upper()}] {msg.content[:300]}")

print()
print("=== CRITIQUE PROMPT ===")
rendered_critique = critique_prompt.format_messages(
    question=debug_q,
    answer=debug_answer,
)
for msg in rendered_critique:
    print(f"  [{msg.type.upper()}] {msg.content[:300]}")

In [ ]:
# ─── 7-B: Experiment -- how MAX_ITERATIONS affects quality and cost ────────────
# Run the same question with caps of 1, 2, and 3 iterations.
# Compare: does the answer meaningfully improve? When does it plateau?

experiment_q = "What is the Reflexion technique and how does it improve LLM reasoning?"
print(f"Question: '{experiment_q}'\n")

for cap in [1, 2, 3]:
    def make_router(max_iters):
        def _router(state: ReflexionState) -> Literal["generate", "__end__"]:
            if state["confident"] or state["iterations"] >= max_iters:
                return END
            return "generate"
        return _router

    g = StateGraph(ReflexionState)
    g.add_node("generate", generate)
    g.add_node("critique", critique)
    g.add_edge(START, "generate")
    g.add_edge("generate", "critique")
    g.add_conditional_edges("critique", make_router(cap))
    temp_app = g.compile()

    result = temp_app.invoke(make_initial_state(experiment_q))
    print(f"cap={cap}:  iters={result['iterations']}  confident={result['confident']}")
    print(f"  answer ({len(result['answer'])} chars): {result['answer'][:240]}...")
    print()

print("Observation: beyond cap=2 or cap=3, quality rarely improves substantially.")
print("Diminishing returns set in quickly for factual questions.")

---

## Part 8 ★ — Extensions (Bonus)

---

### Score History Tracking

Add a `score_history: list` field to `ReflexionState`. Update the `critique` node to append the confidence score each round. After the loop, the history shows how confidence evolved — e.g., `['low', 'medium', 'high']`.

### LangGraph Checkpointing

Compile the graph with an in-memory checkpointer to enable:
- Pausing and resuming the loop at any step
- Human-in-the-loop: a human can review the critique and override before the revision
- Full state history inspectable after the run

```python
from langgraph.checkpoint.memory import MemorySaver
app = graph.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "reflexion-demo"}}
result = app.invoke(make_initial_state(question), config=config)
history = list(app.get_state_history(config))  # every checkpoint
```

### Grounded Reflexion

Extend the `generate` node to retrieve documents from a vector store before drafting. Update `critique_prompt` to check whether the answer is grounded in the retrieved context. This combines the Reflexion loop with RAG — see example 25 (Adaptive RAG) for the retrieval patterns to reuse.

In [ ]:
# ─── 8-A: Track score history across iterations ───────────────────────────────
# Extend state with a score_history list and observe convergence.


class ReflexionStateWithHistory(TypedDict):
    question: str
    answer: str
    critique: str
    iterations: int
    confident: bool
    score_history: Annotated[list, operator.add]  # LangGraph concatenates lists


def critique_with_history(state: ReflexionStateWithHistory) -> dict:
    result = (critique_prompt | evaluator).invoke(
        {"question": state["question"], "answer": state["answer"]}
    )
    return {
        "critique": result.critique,
        "confident": result.score == "high",
        "score_history": [result.score],  # appended via Annotated operator.add
    }


hist_graph = StateGraph(ReflexionStateWithHistory)
hist_graph.add_node("generate", generate)
hist_graph.add_node("critique", critique_with_history)
hist_graph.add_edge(START, "generate")
hist_graph.add_edge("generate", "critique")
hist_graph.add_conditional_edges(
    "critique",
    lambda s: END if (s["confident"] or s["iterations"] >= MAX_ITERATIONS) else "generate"
)
hist_app = hist_graph.compile()

hist_q = "What are the main advantages of LangGraph over vanilla LangChain for building agents?"
hist_result = hist_app.invoke(
    {
        "question": hist_q,
        "answer": "",
        "critique": "",
        "iterations": 0,
        "confident": False,
        "score_history": [],
    }
)

print(f"Score progression : {hist_result['score_history']}")
print(f"Total iterations  : {hist_result['iterations']}")
print(f"Final confidence  : {hist_result['confident']}")
print()
print("A path like ['low', 'medium', 'high'] shows healthy convergence.")
print("A path like ['low', 'low', 'low'] signals the question is unanswerable or the prompt needs tuning.")

In [ ]:
# ─── 8-B: Add a checkpointer and inspect state history ───────────────────────

try:
    from langgraph.checkpoint.memory import MemorySaver

    ckpt_graph = StateGraph(ReflexionState)
    ckpt_graph.add_node("generate", generate)
    ckpt_graph.add_node("critique", critique)
    ckpt_graph.add_edge(START, "generate")
    ckpt_graph.add_edge("generate", "critique")
    ckpt_graph.add_conditional_edges("critique", should_continue)

    ckpt_app = ckpt_graph.compile(checkpointer=MemorySaver())
    config = {"configurable": {"thread_id": "reflexion-workshop-demo"}}

    q = "What is the Reflexion technique and how does it improve LLM reasoning?"
    result = ckpt_app.invoke(make_initial_state(q), config=config)

    print(f"Final state: iters={result['iterations']}  confident={result['confident']}")
    print()

    # Inspect the full state history (one entry per node execution)
    history = list(ckpt_app.get_state_history(config))
    print(f"Checkpoint history: {len(history)} entries")
    for i, h in enumerate(reversed(history)):
        vals = h.values
        print(
            f"  [{i}] iters={vals.get('iterations', 0)}  "
            f"confident={vals.get('confident', False)}  "
            f"answer_len={len(vals.get('answer', ''))}"
        )

except ImportError as e:
    print(f"Checkpointer not available in this LangGraph version: {e}")
    print("Upgrade: pip install langgraph --upgrade")

---

## Exercises

---

### Exercise 1 — Tune `MAX_ITERATIONS`

Change `MAX_ITERATIONS` to `5`. Re-run the easy questions. Do answers meaningfully improve beyond iteration 3, or does the model converge earlier? Use the `score_history` pattern from Part 8 to visualise where quality plateaus.

**Goal:** find the "sweet spot" cap for your specific question domain where quality gain per iteration becomes negligible.

### Exercise 2 — Stricter Critique Prompt

Rewrite `critique_prompt` to require the model to explicitly list specific factual claims it **cannot verify**. Observe how this changes revision behaviour — does a stricter evaluator force more iterations? Does the final answer quality improve?

### Exercise 3 — Human-in-the-Loop Override

Compile the graph with a `MemorySaver` checkpointer and an **interrupt** after the `critique` node. When the loop pauses, print the critique and ask the user to accept it or provide their own. Resume the graph with the human-edited critique.

**Hint:** use `graph.compile(checkpointer=MemorySaver(), interrupt_after=["critique"])`

### Exercise 4 — Grounded Reflexion (Advanced)

Extend the `generate` node to retrieve documents from a Chroma vectorstore (reuse the setup from example 12 or 17). Update `critique_prompt` to check whether the answer is grounded in the retrieved context. Compare answer quality against the ungrounded version on factual questions.

### Exercise 5 — Async Reflexion

LangGraph supports async natively. Rewrite `generate` and `critique` as `async def` functions. Compile the same graph and call `await app.ainvoke(...)` inside an async context. Measure whether async execution reduces wall-clock time on multiple concurrent questions.

In [ ]:
# ── Exercise 1 Starter — tune MAX_ITERATIONS ──────────────────────────────────

# TODO: change this value and rebuild the graph
MY_MAX_ITERATIONS = 5

# TODO: rebuild the graph with MY_MAX_ITERATIONS as the cap
# TODO: run the same question at cap=1, 2, 3, 4, 5
# TODO: print score_history for each cap and compare final answer quality

print(f"Target: MAX_ITERATIONS = {MY_MAX_ITERATIONS}")
print("TODO: rebuild graph and run experiment")

In [ ]:
# ── Exercise 2 Starter — stricter critique prompt ─────────────────────────────

# TODO: replace the system message with a stricter evaluation rubric
STRICT_CRITIQUE_SYSTEM = """
Evaluate this answer rigorously:
1. List each factual claim in the answer.
2. For each claim, state whether you can verify it with certainty or not.
3. List any important information missing from the answer.
4. Rate your overall confidence: low / medium / high.
   Only rate 'high' if ALL claims are verifiable and no important gaps remain.
"""

strict_critique_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", STRICT_CRITIQUE_SYSTEM),
        ("human", "Question: {question}\n\nAnswer: {answer}"),
    ]
)

# TODO: rebuild the critique node using strict_critique_prompt
# TODO: compile a new graph and compare iteration counts vs the lenient prompt

print("Strict critique prompt defined.")
print("TODO: wire into a new graph and compare results")

---

## What's Next?

You now have a complete foundation in the Reflexion pattern. Here's where to go deeper:

### Apply it to harder problems
- **Example 17 — Corrective RAG** ([`../17-corrective-rag/`](../17-corrective-rag/)): add a retrieval step inside `generate` so the actor grounds its answer in real documents before the evaluator checks it.
- **Example 25 — Adaptive RAG** ([`../25-adaptive-rag/`](../25-adaptive-rag/)): route queries dynamically across no-retrieval / vectorstore / web-search paths — combine with Reflexion for grounded self-correction.

### Go further with LangGraph
- **Example 28 — Parallel Subgraphs** ([`../28-parallel-subgraphs/`](../28-parallel-subgraphs/)): use `Send()` to fan out to multiple agents in parallel, then collect and rank their outputs.
- **Example 30 — Agentic RAG** ([`../30-agentic-rag/`](../30-agentic-rag/)): a full agentic RAG pipeline with tool calls, grading, and web fallback.

### Measure what you built
- **Example 16 — RAG Evaluation with RAGAS** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): score faithfulness, answer relevance, and context recall automatically.

### Further reading
- Shinn et al. (2023). *Reflexion: Language Agents with Verbal Reinforcement Learning.* NeurIPS 2023. https://arxiv.org/abs/2303.11366
- Yao et al. (2023). *ReAct: Synergizing Reasoning and Acting in Language Models.* ICLR 2023. https://arxiv.org/abs/2210.03629
- Asai et al. (2023). *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection.* https://arxiv.org/abs/2310.11511
- LangGraph concepts (state, edges, checkpointing): https://langchain-ai.github.io/langgraph/concepts/
- LangGraph how-to guides: https://langchain-ai.github.io/langgraph/how-tos/

---

*Workshop complete. The natural next step is example 25 — add dynamic retrieval routing on top of the self-correction loop you just built.*

---

## Exercise Answer Key

Use this section as a reference **after** attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 — Tune `MAX_ITERATIONS`

**Sample answer:** For well-known factual questions, quality typically plateaus at cap=2 or cap=3. Increasing to 5 produces diminishing returns: the model may rephrase the same correct answer without meaningful improvement. The `score_history` will show `['medium', 'high']` and early exit rather than running all 5 iterations.

**What good output looks like:**
- `score_history = ['medium', 'high']` — confident=True at iteration 2, loop exits early
- `score_history = ['low', 'medium', 'high']` — 3 iterations needed, all productive
- `score_history = ['low', 'low', 'low', 'low', 'low']` — question is unanswerable, hits cap

**Rule of thumb:** `MAX_ITERATIONS = 3` is a good default for most Q&A tasks. Raise to 5 only for complex reasoning or code generation tasks where multi-step improvement is expected.

In [ ]:
# Sample solution for Exercise 1
# Build graphs with cap=1..5 and report score_history for a known-good question

ex1_q = "What are the main advantages of LangGraph over vanilla LangChain for building agents?"

for cap in range(1, 6):
    def _make_router(max_i):
        def _r(state):
            return END if (state["confident"] or state["iterations"] >= max_i) else "generate"
        return _r

    g = StateGraph(ReflexionStateWithHistory)
    g.add_node("generate", generate)
    g.add_node("critique", critique_with_history)
    g.add_edge(START, "generate")
    g.add_edge("generate", "critique")
    g.add_conditional_edges("critique", _make_router(cap))
    a = g.compile()

    r = a.invoke(
        {
            "question": ex1_q, "answer": "", "critique": "",
            "iterations": 0, "confident": False, "score_history": []
        }
    )
    print(f"cap={cap}: iters={r['iterations']}  history={r['score_history']}  confident={r['confident']}")

### Exercise 2 — Stricter Critique Prompt

**Sample answer:** A stricter critique prompt that requires claim-by-claim verification forces the evaluator to be more discerning. Typically this increases iteration counts by 1-2 for well-known factual questions — the evaluator identifies nuances the lenient prompt missed. For unanswerable questions, the strict prompt is more decisive about scoring `low` early rather than `medium`, which can actually reduce wasted iterations.

**What to look for:**
- The critique text should list individual claims, not just a general assessment
- Iteration count may increase for marginal questions but final quality should improve
- If iteration count increases dramatically on easy questions, the prompt is too strict — recalibrate the `high` threshold description

### Exercise 3 — Human-in-the-Loop Override

**Sample answer:** `interrupt_after=["critique"]` pauses the graph after each critique node. Use `app.get_state(config)` to read the current critique, modify it with `app.update_state(config, {"critique": my_critique})`, then `app.invoke(None, config=config)` to resume. This pattern lets a human reviewer correct hallucinated critiques before the model revises.

### Exercise 4 — Grounded Reflexion

**Sample answer:** Extend `generate` to call `retriever.invoke(state["question"])` and prepend the retrieved chunks to the prompt context. Update `critique_prompt` to add: *"Check whether each claim in the answer is supported by the provided context."* Grounded Reflexion outperforms ungrounded Reflexion on factual retrieval tasks because the evaluator now has an external reference — not just the model's own prior knowledge — to judge against.

### Exercise 5 — Async Reflexion

**Sample answer:** Rename `generate` to `agenerate` and add `async def`. Replace `(prompt | llm).invoke(...)` with `await (prompt | llm).ainvoke(...)`. Compile the same graph structure and call `await app.ainvoke(...)` in an `asyncio.run()` block or inside an async notebook cell. Async is most beneficial when running many questions concurrently — use `asyncio.gather(*[app.ainvoke(make_initial_state(q)) for q in questions])` to fan out.